In [1]:
! pip install numpy
! pip install scipy

In [2]:
"""
Linear Algebra robust example, reading A from a .txt file.

Usage:
    - Put your matrix in a text file, e.g. "hw2_data7.txt"
      Each row on its own line, values separated by spaces or tabs.
    - Run: python hw2a_txt_numpy_only.py

Notes:
    - Only depends on numpy.
    - Implements LU with partial pivoting in pure numpy.
"""

import numpy as np
from scipy.linalg import lu
import time
import sys

In [3]:

# ---------- Utility linear algebra helpers (numpy-only) ----------
def is_symmetric(A, tol=1e-12):
    n,m=np.shape(A)
    if n!=m:
        return False
    return np.allclose(A, A.T, atol=tol, rtol=0)

def is_positive_definite(A):
    # Test via Cholesky (NumPy raises LinAlgError if not PD)
    try:
        np.linalg.cholesky(A)
        return True
    except np.linalg.LinAlgError:
        return False

In [4]:
# ---------- Robust forward / backward substitution helpers ----------
def forward_substitution(L, b, tol=1e-16):
    """
    Solve L x = b where L is lower-triangular (may have unit diagonal).
    Returns 1-D x of length = number of columns of L.
    """
    b = np.asarray(b).ravel()
    m, k = L.shape
    if b.size != m:
        raise ValueError(f"Incompatible shapes: L has {m} rows but b has length {b.size}")
    x = np.zeros(k, dtype=L.dtype)

    for i in range(m):
        if i == 0:
            s = 0.0
        else:
            s = L[i, :i] @ x[:i]
        # diagonal element might be 1 for LU's L
        diag = L[i, i] if i < k else 0.0
        if abs(diag) <= tol:
            # treat unit-diagonal as 1
            if np.isclose(diag, 1.0, atol=1e-12):
                diag = 1.0
            else:
                raise np.linalg.LinAlgError(f"Zero (or tiny) diagonal at L[{i},{i}] = {diag}")
        x[i] = (b[i] - s) / diag
    return x[:k]


def backward_substitution(U, b, tol=1e-16):
    """
    Solve U x = b where U is upper-triangular.
    U shape (m,n), b length m. Returns x of length n.
    """
    b = np.asarray(b).ravel()
    m, n = U.shape
    if b.size != m:
        raise ValueError(f"Incompatible shapes: U has {m} rows but b has length {b.size}")
    x = np.zeros(n, dtype=U.dtype)

    # iterate over row indices from last row to first
    for row in range(m - 1, -1, -1):
        diag_col = row
        if diag_col >= n:
            raise ValueError(f"U has shape {U.shape}, cannot access diagonal at column {diag_col}")
        if diag_col + 1 < n:
            s = U[row, diag_col+1:] @ x[diag_col+1:]
        else:
            s = 0.0
        diag = U[row, diag_col]
        if abs(diag) <= tol:
            raise np.linalg.LinAlgError(f"Zero (or tiny) diagonal at U[{row},{diag_col}] = {diag}")
        x[diag_col] = (b[row] - s) / diag

    return x

In [5]:


# ---------- Updated solve_and_compare using forward/backward solves ----------
def solve_and_compare(A, b, x_true, methods):
    m, n = A.shape
    x_true = x_true.reshape(-1)  # 1D for norm computations
    b = np.asarray(b).ravel()    # ensure 1-D

    for method in methods:
        start = time.perf_counter()
        try:
            if method == 'LU':
                # Use SciPy lu (returns P,L,U with A = P @ L @ U)
                P, L, U = lu(A)
                print("shapes:", P.shape, L.shape, U.shape, b.shape)
                # Solve L y = P^T b  (forward), then U x = y (backward)
                Pb = P.T @ b
                y = forward_substitution(L, Pb)
                x = backward_substitution(U, y)

            elif method == 'QR':
                # QR works for rectangular. Use reduced economy Q,R
                Q, R = np.linalg.qr(A, mode='reduced')  # Q shape (m,k), R shape (k,n) where k = min(m,n)
                Qtb = Q.T @ b
                # If R is square upper triangular (when m >= n), back-substitute.
                # If R is rectangular (m < n), use least squares on R x = Qtb (via lstsq)
                k = R.shape[0]
                if R.shape[0] == R.shape[1]:
                    # square upper triangular (n x n) -> use backward_substitution
                    x = backward_substitution(R, Qtb)
                else:
                    # underdetermined/wide: solve R x = Qtb with least-squares
                    x, *_ = np.linalg.lstsq(R, Qtb, rcond=None)

            elif method == 'Cholesky':
                if m != n:
                    raise ValueError("Cholesky requires a square matrix.")
                # NumPy returns lower-triangular L such that A = L @ L.T
                L = np.linalg.cholesky(A)
                # Solve L y = b (forward), then L.T x = y (backward)
                y = forward_substitution(L, b)
                x = backward_substitution(L.T, y)

            elif method == 'Eigen':
                if m != n:
                    raise ValueError("Eigen decomposition requires a square matrix.")
                if is_symmetric(A):
                    # Use eigh for symmetric: V orthonormal
                    eigvals, V = np.linalg.eigh(A)
                    if np.any(np.isclose(eigvals, 0.0)):
                        raise np.linalg.LinAlgError("Eigen: zero eigenvalue -> singular or ill-conditioned.")
                    coeffs = (V.T @ b) / eigvals
                    x = V @ coeffs
                else:
                    eigvals, V = np.linalg.eig(A)
                    if np.any(np.isclose(eigvals, 0.0)):
                        raise np.linalg.LinAlgError("Eigen: zero eigenvalue -> singular or ill-conditioned.")
                    Vinv_b = np.linalg.solve(V, b)   # inv(V) @ b
                    z = Vinv_b / eigvals
                    x = V @ z
                    # drop tiny imaginary parts if present
                    if np.iscomplexobj(x) and np.max(np.abs(np.imag(x))) < 1e-12:
                        x = np.real(x)

            elif method == 'SVD':
                # Use economy SVD
                U_svd, s, Vt = np.linalg.svd(A, full_matrices=False)
                y = U_svd.T @ b
                # Tolerance to treat small singular values as zero
                tol = max(A.shape) * np.finfo(float).eps * (s.max() if s.size else 1.0)
                s_inv = np.array([1/si if si > tol else 0.0 for si in s])
                z = s_inv * y
                x = Vt.T @ z

            else:
                raise ValueError(f"Unknown method '{method}'")

            elapsed = time.perf_counter() - start

            err = np.linalg.norm(x.reshape(-1) - x_true)
            res = np.linalg.norm(A @ x.reshape(-1) - b)

            print(f"{method:<10} | error: {err:.2e} | residual: {res:.2e} | time: {elapsed:.3f}s")

        except Exception as e:
            print(f"{method:<10} | FAILED ({e})")


In [6]:
# Change this filename to your .txt file. Example: "hw2_data7.txt"
filename = "hw2_data1.txt"

# Load A from text file. np.loadtxt handles whitespace-separated numeric data.
try:
    A = np.loadtxt(filename, delimiter=',')
except OSError:
    print(f"File '{filename}' not found or could not be opened.")
    sys.exit(1)
except ValueError as ve:
    print(f"Could not parse matrix from '{filename}': {ve}")
    sys.exit(1)

if A.ndim == 1:
    # Single row — interpret as 1 x n
    A = A.reshape(1, -1)

m, n = A.shape
print(f"Matrix is {m} x {n}")

# If you want to see the matrix, you can uncomment this here
print(A)

Matrix is 10 x 10
[[ 0.53766714 -1.34988694  0.67149713  0.88839563 -0.10224245 -0.86365282
  -1.0890643  -0.61560188  1.41931015 -1.14795278]
 [ 1.83388501  3.03492347 -1.20748692 -1.14707011 -0.24144704  0.07735909
   0.03255746  0.74807678  0.29158437  0.10487472]
 [-2.25884686  0.72540422  0.71723865 -1.06887046  0.31920674 -1.21411704
   0.55252702 -0.19241851  0.19781105  0.72225403]
 [ 0.86217332 -0.06305487  1.63023529 -0.80949869  0.3128586  -1.11350074
   1.10061022  0.88861043  1.58769909  2.58549125]
 [ 0.31876524  0.7147429   0.48889377 -2.94428416 -0.86487992 -0.00684933
   1.5442119  -0.76484924 -0.80446596 -0.66689067]
 [-1.3076883  -0.20496606  1.03469301  1.43838029 -0.0300513   1.53263031
   0.08593113 -1.40226897  0.69662442  0.18733102]
 [-0.43359202 -0.12414435  0.72688513  0.32519054 -0.16487902 -0.76966591
  -1.49159031 -1.42237593  0.83508817 -0.08249443]
 [ 0.34262447  1.48969761 -0.30344092 -0.75492832  0.62770729  0.37137881
  -0.74230184  0.48819391 -0.2437

In [7]:

# ground truth x_true = ones(n)
x_true = np.ones(n)
b_true = A @ x_true

# Symmetry / PD checks
if is_symmetric(A):
    print("A is symmetric.")
    if is_positive_definite(A):
        print("A is SPD!")
    else:
        print("A is NOT positive definite.")
else:
    print("A is NOT SPD.")


methods = ['LU', 'QR', 'Cholesky', 'Eigen', 'SVD']
solve_and_compare(A, b_true, x_true, methods)


A is NOT SPD.
shapes: (10, 10) (10, 10) (10, 10) (10,)
LU         | error: 1.27e-15 | residual: 1.84e-15 | time: 0.020s
QR         | error: 1.40e-15 | residual: 5.67e-15 | time: 0.091s
Cholesky   | FAILED (Matrix is not positive definite)
Eigen      | error: 7.58e-15 | residual: 2.36e-14 | time: 0.032s
SVD        | error: 3.65e-15 | residual: 6.48e-15 | time: 0.020s
